# One NavigateToPose goal through Nav2 (RO-080)
**This is our own Python version of Nav2's servers** (`images/nav2sim.py`), written from the navigation2 code on `main` (read 2026-10-12) and the bring-up parameters in `nav2_bringup/params/nav2_params.yaml`. **It is not Nav2 running.** DWB comes from RO-078's `dwbkit.py` (copied). Every number in the Note comes from this notebook.

Runs (seed 0): `main` (the Note's goal, bring-up collision monitor), `clear` (no person), `alpha_bringup` (AMCL with the bring-up alphas 0.2), `example_zones` (the collision monitor package's example zones).

In [1]:
import sys, math
from pathlib import Path
import numpy as np
import pandas as pd
HERE = Path.cwd()
sys.path.insert(0, str(HERE / "images"))
import nav2sim as N
import dwbkit as D
DATA = HERE / "data"; DATA.mkdir(exist_ok=True)
R = N.results()
for k, r in R.items():
    L = r["log"]
    e = [np.hypot(*(x["map"][:2] - x["true"][:2])) for x in L]
    print(f"{k:14s} {r['result']:28s} t = {r['t_final']:.2f} s  plans {[(round(t, 2), round(D.path_length(p), 3)) for t, p in r['plans']]}"
          f"  AMCL error mean {np.mean(e):.3f} max {np.max(e):.3f} m")

main           goal reached                 t = 11.51 s  plans [(0.0, 3.26), (3.0, 2.986)]  AMCL error mean 0.035 max 0.054 m
clear          goal reached                 t = 9.46 s  plans [(0.0, 3.26)]  AMCL error mean 0.050 max 0.097 m
alpha_bringup  goal reached                 t = 11.11 s  plans [(0.0, 3.26), (3.0, 2.937), (11.0, 0.241)]  AMCL error mean 0.212 max 0.447 m
example_zones  failed to make progress      t = 12.45 s  plans [(0.0, 3.26), (3.0, 3.022)]  AMCL error mean 0.025 max 0.031 m


## 1. Bring-up: the lifecycle manager

In [2]:
B = N.lifecycle_bringup()
print(len(N.NODES), "nodes;", len(B), "transitions")
print("configure order:", [n for n, tr, s in B if tr == "configure"])
print("first activate:", [b for b in B if b[1] == "activate"][0])

17 nodes; 34 transitions
configure order: ['map_server', 'amcl', 'keepout_filter_mask_server', 'keepout_costmap_filter_info_server', 'speed_filter_mask_server', 'speed_costmap_filter_info_server', 'controller_server', 'smoother_server', 'planner_server', 'route_server', 'behavior_server', 'velocity_smoother', 'collision_monitor', 'bt_navigator', 'waypoint_follower', 'docking_server', 'following_server']
first activate: ('map_server', 'activate', 'active')


## 2. The map and AMCL

In [3]:
occ, unk = N.build_map()
print("map cells", occ.shape[1], "x", occ.shape[0], " occupied", int(occ.sum()), " unknown", int(unk.sum()))
m = R["main"]
U = m["amcl"]
print("AMCL updates:", len(U), " particles after each:", [u["n_after"] for u in U][:6], "...")
print("update times:", [round(u["t"], 2) for u in U])
L = m["log"]
err = np.array([np.hypot(*(x["map"][:2] - x["true"][:2])) for x in L])
oerr = np.array([np.hypot(*(x["odom"][:2] - x["true"][:2])) for x in L])
herr = np.degrees([abs(D.wrap(x["odom"][2] - x["true"][2])) for x in L])
print(f"main: AMCL error mean {err.mean():.3f} max {err.max():.3f} m; odometry error at the end {oerr[-1]:.3f} m, heading {herr[-1]:.1f} deg")
times = np.array([x["t"] for x in L])
for j in (0, 2, 10, 20):
    k = int(np.argmin(abs(times - U[j]["t"])))
    print(f"update {j + 1} at {U[j]['t']:.1f} s: {U[j]['n_after']} particles; odometry off {oerr[k]:.3f} m, AMCL off {err[k]:.3f} m")
La = R["alpha_bringup"]["log"]
ea = np.array([np.hypot(*(x["map"][:2] - x["true"][:2])) for x in La])
print(f"bring-up alphas 0.2: AMCL error mean {ea.mean():.3f} max {ea.max():.3f} m; particles at the end {R['alpha_bringup']['amcl'][-1]['n_after']}")
# the likelihood field's per-beam term at easy values
zr = 0.5 / 30
for z in (0.0, 0.1, 0.4):
    pz = 0.5 * math.exp(-z * z / (2 * 0.2 ** 2)) + zr
    print(f"z = {z} m: pz = {pz:.4f}, pz^3 = {pz ** 3:.4f}")
# KLD limit for a few bin counts
a = N.AMCL(occ, N.START.copy(), np.random.default_rng(0))
print("KLD limit:", {k: a.kld_limit(k) for k in (2, 10, 50, 100)})
pd.DataFrame([dict(t=u["t"], x=u["est"][0], y=u["est"][1], th=u["est"][2], n=u["n_after"], ox=u["odom"][0], oy=u["odom"][1]) for u in U]).to_csv(DATA / "amcl_updates.csv", index=False)

map cells 160 x 100  occupied 2547  unknown 4580
AMCL updates: 21  particles after each: [2000, 2000, 501, 501, 501, 501] ...
update times: [0.0, 0.5, 1.0, 1.6, 2.1, 2.6, 3.3, 3.7, 4.3, 4.6, 4.9, 5.2, 5.6, 6.0, 6.5, 7.3, 8.8, 9.2, 9.7, 10.3, 11.3]
main: AMCL error mean 0.035 max 0.054 m; odometry error at the end 0.222 m, heading 9.3 deg
update 1 at 0.0 s: 2000 particles; odometry off 0.000 m, AMCL off 0.000 m
update 3 at 1.0 s: 501 particles; odometry off 0.002 m, AMCL off 0.001 m
update 11 at 4.9 s: 501 particles; odometry off 0.107 m, AMCL off 0.041 m
update 21 at 11.3 s: 501 particles; odometry off 0.222 m, AMCL off 0.048 m
bring-up alphas 0.2: AMCL error mean 0.212 max 0.447 m; particles at the end 1471
z = 0.0 m: pz = 0.5167, pz^3 = 0.1379
z = 0.1 m: pz = 0.4579, pz^3 = 0.0960
z = 0.4 m: pz = 0.0843, pz^3 = 0.0006
KLD limit: {2: 500, 10: 500, 50: 588, 100: 1129}


## 3. Costmaps

In [4]:
for d in (0.0, 0.2, 0.25, 0.30, 0.50, 0.70, 0.75):
    lethal = np.zeros((41, 41), bool); lethal[20, 20] = True
    c = N.inflation(lethal)
    i = int(round(d / N.RES))
    print(f"cell {d:.2f} m from an obstacle cell: cost {c[20, 20 + i] if 20 + i < 41 else 0}")
print("formula at 0.30:", 252 * math.exp(-3.0 * (0.30 - 0.25)), " at 0.50:", 252 * math.exp(-3.0 * 0.25), " at 0.70:", 252 * math.exp(-3.0 * 0.45))
keepcycles = {round(x["t"], 2): x for x in L if x["gcm"] is not None}
print("snapshots at", sorted(keepcycles))
for t in (2.4, 3.0):
    x = keepcycles[t]
    lc, ox, oy, ll = x["lcm"]
    gc, gl = x["gcm"]
    pc = np.hypot(*(np.array(N.D.PERSON) - 0)) if False else None
    # person cells marked: lethal cells within 0.3 m of the person's centre
    def person_cells(lethal, ox0, oy0):
        J, I = np.nonzero(lethal)
        X, Y = ox0 + (I + 0.5) * N.RES, oy0 + (J + 0.5) * N.RES
        return int(np.sum(np.hypot(X - D.PERSON[0], Y - D.PERSON[1]) < 0.3))
    print(f"t = {t}: local costmap {lc.shape}, person cells marked locally {person_cells(ll, ox, oy)}, "
          f"globally {person_cells(gl, N.OX, N.OY)}")

cell 0.00 m from an obstacle cell: cost 254
cell 0.20 m from an obstacle cell: cost 253
cell 0.25 m from an obstacle cell: cost 253
cell 0.30 m from an obstacle cell: cost 216
cell 0.50 m from an obstacle cell: cost 119
cell 0.70 m from an obstacle cell: cost 65
cell 0.75 m from an obstacle cell: cost 0
formula at 0.30: 216.89841005911455  at 0.50: 119.0363712907357  at 0.70: 65.32854568276466
snapshots at [0.0, 1.0, 2.4, 2.55, 3.0]
t = 2.4: local costmap (60, 60), person cells marked locally 9, globally 0
t = 3.0: local costmap (60, 60), person cells marked locally 11, globally 11


## 4. The behaviour tree and the planner

In [5]:
for e in m["events"][:6]:
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in e.items()})
print("... every check from 4.0 s on:", sorted({e['status'] for e in m['events'] if e['t'] >= 4.0}))
print("NavFn cost for costmap 216:", 50 + 0.8 * 216, " for 0:", 50, " for 252:", 50 + 0.8 * 252)
P0 = m["plans"][0][1]; P1 = m["plans"][1][1]
print(f"first plan {D.path_length(P0):.3f} m, {len(P0)} poses; replan at {m['plans'][1][0]} s: {D.path_length(P1):.3f} m")
pd.DataFrame(P0, columns=["x", "y", "th"]).to_csv(DATA / "plan0.csv", index=False)
pd.DataFrame(P1, columns=["x", "y", "th"]).to_csv(DATA / "plan1.csv", index=False)

{'t': 0.0, 'node': 'ComputePathToPose', 'status': 'SUCCESS', 'why': 'no path yet', 'length': 3.26, 'reached': 7600}
{'t': 1.0, 'node': 'CheckIfNewPathNeeded', 'status': 'SUCCESS', 'remaining': 3.025, 'valid': True, 'blocked_index': None}
{'t': 2.0, 'node': 'CheckIfNewPathNeeded', 'status': 'SUCCESS', 'remaining': 2.45, 'valid': True, 'blocked_index': None}
{'t': 3.0, 'node': 'CheckIfNewPathNeeded', 'status': 'FAILURE', 'remaining': 2.0, 'valid': False, 'blocked_index': 68}
{'t': 3.0, 'node': 'ComputePathToPose', 'status': 'SUCCESS', 'why': 'path blocked on the global costmap', 'length': 2.986, 'reached': 7358}
{'t': 4.0, 'node': 'CheckIfNewPathNeeded', 'status': 'SUCCESS', 'remaining': 2.986, 'valid': True, 'blocked_index': None}
... every check from 4.0 s on: ['SUCCESS']
NavFn cost for costmap 216: 222.8  for 0: 50  for 252: 251.60000000000002
first plan 3.260 m, 130 poses; replan at 3.0 s: 2.986 m


## 5. Controller, smoother and monitor

In [6]:
c1 = keepcycles[1.0]
rec = c1["rec"]
print(f"t = 1.0 s: speed window {np.round(rec['vwin'], 3)}, turn window {np.round(rec['wwin'], 3)}; pairs {len(rec['pairs'])}, legal {rec['n_legal']}; "
      f"best (v, w) = {np.round(rec['cmd'], 3)}, total {rec['total']:.2f}")
print("cmd_nav", np.round(c1["cmd_nav"], 4), "cmd_smoothed", np.round(c1["cmd_smooth"], 4), "cmd_vel", np.round(c1["cmd_out"], 4))
c0 = L[0]
print("t = 0: cmd_nav", np.round(c0["cmd_nav"], 4), "-> smoothed", np.round(c0["cmd_smooth"], 4))
print("0.5 / 20 =", 0.5 / 20, " 3.2 / 20 =", 3.2 / 20)
print("main run: the monitor acted", sum(1 for x in L if x["zone"]), "times")
gap_t = min(((x["t"], np.hypot(x["true"][0] - D.PERSON[0], x["true"][1] - D.PERSON[1])) for x in L if x["t"] >= D.T_PERSON), key=lambda g: g[1])
print("closest at t =", gap_t[0])
gap = min(np.hypot(x["true"][0] - D.PERSON[0], x["true"][1] - D.PERSON[1]) for x in L if x["t"] >= D.T_PERSON) - D.PERSON_R - D.R_BODY
print(f"closest gap between the robot's disk and the person: {gap:.3f} m")
for x in L:
    if x["zone"]:
        print(f"main run t = {x['t']:.2f}: {x['zone']}, time to collision {x['tc']:.1f} s, factor {x['tc'] / 1.2:.4f}, "
              f"smoothed {np.round(x['cmd_smooth'], 4)} -> {np.round(x['cmd_out'], 4)}, robot {np.round(x['true'], 3)}, "
              f"{np.hypot(*(x['true'][:2] - D.PERSON)):.3f} m from the person's centre")
Lz = R["example_zones"]["log"]
first = [x for x in Lz if x["zone"]][0]
stop = [x for x in Lz if x["zone"] and x["zone"].endswith("stop")][0]
print(f"example zones: first act {first['t']:.2f} s by {first['zone']}; first stop {stop['t']:.2f} s; result {R['example_zones']['result']} at {R['example_zones']['t_final']:.2f} s")
print("stop position", np.round(Lz[-1]["true"], 3))
gap = np.hypot(*(np.array(D.PERSON[:2]) - stop["true"][:2])) - D.PERSON_R - D.R_BODY
corner = np.hypot(0.3, 0.3)
Zs = {z["name"]: z for z in N.MONITOR_EXAMPLE}
print("returns inside the stop box at the first stop:", N.points_inside(Zs["PolygonStop"]["poly"], stop["scan_base"]))
print(f"at the first stop the disks are {gap:.3f} m apart; the stop box's front corners are {corner:.3f} m from the robot's centre, "
      f"{corner - D.R_BODY:.3f} m beyond its edge")

t = 1.0 s: speed window [0.  0.6], turn window [-1.  1.]; pairs 375, legal 373; best (v, w) = [0.568 0.263], total 16.00
cmd_nav [0.5684 0.2632] cmd_smoothed [0.525  0.2632] cmd_vel [0.525  0.2632]
t = 0: cmd_nav [0.4737 0.6842] -> smoothed [0.025 0.16 ]
0.5 / 20 = 0.025  3.2 / 20 = 0.16
main run: the monitor acted 1 times
closest at t = 4.25
closest gap between the robot's disk and the person: 0.039 m
main run t = 2.55: FootprintApproach, time to collision 1.1 s, factor 0.9167, smoothed [0.4684 0.2632] -> [0.4294 0.2412], robot [2.489 1.989 1.364], 0.865 m from the person's centre
example zones: first act 2.55 s by FootprintApproach; first stop 3.70 s; result failed to make progress at 12.45 s
stop position [2.512 2.315 1.641]
returns inside the stop box at the first stop: 5
at the first stop the disks are 0.149 m apart; the stop box's front corners are 0.424 m from the robot's centre, 0.174 m beyond its edge


## 6. The whole goal

In [7]:
end = L[-1]
print(f"goal checker accepts at {L[-1]['t']:.2f} s, result to the client at {m['t_final']:.2f} s; "
      f"clear room: goal checker accepts at {R['clear']['log'][-1]['t']:.2f} s; delay {L[-1]['t'] - R['clear']['log'][-1]['t']:.2f} s")
dg = np.hypot(end["true"][0] - D.GOAL[0], end["true"][1] - D.GOAL[1])
dge = np.hypot(end["map"][0] - D.GOAL[0], end["map"][1] - D.GOAL[1])
print(f"final true pose {np.round(end['true'], 3)}: {dg:.3f} m and {math.degrees(abs(D.wrap(end['true'][2] - D.GOAL[2]))):.1f} deg from the goal; AMCL's estimate {dge:.3f} m")
t0 = [x for x in L if x["t"] > 7 and abs(x["v"]) < 0.005][0]
P1h = m["plans"][1][1]
dd = P1h[-1, :2] - P1h[-8, :2]
print(f"plan's last 0.35 m heads {math.degrees(math.atan2(dd[1], dd[0])):.1f} deg; the robot stops moving forward at {t0['t']:.2f} s facing "
      f"{math.degrees(t0['true'][2]):.1f} deg and turns for {L[-1]['t'] - t0['t']:.2f} s to {math.degrees(L[-1]['true'][2]):.1f} deg")
v = np.array([x["v"] for x in L])
print(f"top speed {v.max():.3f} m/s; drive length {np.sum(np.hypot(*np.diff(np.array([x['true'][:2] for x in L]), axis=0).T)):.3f} m")
pd.DataFrame([dict(t=x["t"], x=x["true"][0], y=x["true"][1], th=x["true"][2], mx=x["map"][0], my=x["map"][1], ox=x["odom"][0], oy=x["odom"][1],
                   v=x["v"], w=x["w"], nav_v=x["cmd_nav"][0], nav_w=x["cmd_nav"][1], sm_v=x["cmd_smooth"][0], sm_w=x["cmd_smooth"][1],
                   out_v=x["cmd_out"][0], out_w=x["cmd_out"][1]) for x in L]).to_csv(DATA / "main_log.csv", index=False)
assert m["result"] == "goal reached" and len(m["plans"]) == 2
print("all checks passed")

goal checker accepts at 11.50 s, result to the client at 11.51 s; clear room: goal checker accepts at 9.45 s; delay 2.05 s
final true pose [3.352 3.598 1.362]: 0.267 m and 12.0 deg from the goal; AMCL's estimate 0.230 m
plan's last 0.35 m heads 6.8 deg; the robot stops moving forward at 8.35 s facing 4.9 deg and turns for 3.15 s to 78.0 deg
top speed 0.593 m/s; drive length 3.301 m
all checks passed
